# Bronze Layer Ingestion Notebook
Reads manually downloaded Kaggle files from the Data folder and creates timestamped Bronze copies.

## Data Source and Ingestion Approach

The initial plan for this project was to automate the data download process using the Kaggle API. The API was successfully configured and authenticated; however, due to organisational security policies and access restrictions within the working environment, the dataset could not be downloaded using this approach.

To ensure the project could be completed in a secure and compliant manner, the dataset was downloaded manually from Kaggle using the organisation's approved web browser and saved to the project data folder. The Bronze ingestion process then reads these source files directly and creates timestamped Bronze copies for further processing.

This approach still follows the principles of Medallion Architecture by preserving original raw data before it is transformed and refined within the Silver and Gold layers.

### Notebook Flow

1. Configure paths and ingestion metadata.
2. Validate required source files are present.
3. Load source datasets.
4. Append snapshot metadata columns to tabular data.
5. Write timestamped Bronze outputs.

In [2]:
# =========================================================================
# Bronze Ingestion Script
# Creates a timestamped snapshot of raw source data in the Bronze folder.
# No transformations are applied at this stage.
# =========================================================================

from datetime import datetime
from pathlib import Path
import json

import pandas as pd

# Resolve the Data directory for both common launch locations:
# - Assessment/ (cwd contains Data)
# - Assessment/Scripts/ (cwd parent contains Data)
current_dir = Path.cwd().resolve()
candidate_data_dirs = [
    current_dir / "Data",
    current_dir.parent / "Data",
]
DATA_DIR = next((path for path in candidate_data_dirs if path.exists()), candidate_data_dirs[0])
if not DATA_DIR.exists():
    checked = "\n - ".join(str(path) for path in candidate_data_dirs)
    raise FileNotFoundError(
        f"Data directory not found. Checked:\n - {checked}"
    )

BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

ingestion_time = datetime.now()
snapshot_date = ingestion_time.strftime("%Y%m%d")
ingestion_timestamp = ingestion_time.isoformat(timespec="seconds")

print(f"Snapshot Date: {snapshot_date}")
print(f"Ingestion Timestamp: {ingestion_timestamp}")

Snapshot Date: 20260825
Ingestion Timestamp: 2026-08-25T22:04:43


In [3]:
print("Current working directory:", current_dir)
print("Data directory:", DATA_DIR)
print("Bronze directory:", BRONZE_DIR)

Current working directory: C:\Users\whale\BSc\DSPP\Scripts
Data directory: C:\Users\whale\BSc\DSPP\Data
Bronze directory: C:\Users\whale\BSc\DSPP\Data\bronze


In [4]:
SOURCE_FILES = {
    "users": DATA_DIR / "users_data.csv",
    "cards": DATA_DIR / "cards_data.csv",
    "transactions": DATA_DIR / "transactions_data.csv",
    "mcc_codes": DATA_DIR / "mcc_codes.json",
    "fraud_labels": DATA_DIR / "train_fraud_labels.json",
}

print("Source files:")
for name, path in SOURCE_FILES.items():
    print(f" - {name:12} | exists={path.exists()} | {path.name}")

missing_files = [
    str(path)
    for path in SOURCE_FILES.values()
    if not path.exists()
]
if missing_files:
    missing_text = "\n - ".join(missing_files)
    raise FileNotFoundError(
        f"Required source files are missing:\n - {missing_text}"
    )

Source files:
 - users        | exists=True | users_data.csv
 - cards        | exists=True | cards_data.csv
 - transactions | exists=True | transactions_data.csv
 - mcc_codes    | exists=True | mcc_codes.json
 - fraud_labels | exists=True | train_fraud_labels.json


In [5]:
users_df = pd.read_csv(SOURCE_FILES["users"])
cards_df = pd.read_csv(SOURCE_FILES["cards"])
transactions_df = pd.read_csv(SOURCE_FILES["transactions"])

with SOURCE_FILES["mcc_codes"].open("r", encoding="utf-8") as file:
    mcc_data = json.load(file)

with SOURCE_FILES["fraud_labels"].open("r", encoding="utf-8") as file:
    fraud_labels_data = json.load(file)

for dataframe in [users_df, cards_df, transactions_df]:
    dataframe["snapshot_date"] = snapshot_date
    dataframe["ingestion_timestamp"] = ingestion_timestamp

print(f"Users rows: {len(users_df):,}")
print(f"Cards rows: {len(cards_df):,}")
print(f"Transactions rows: {len(transactions_df):,}")

Users rows: 2,000
Cards rows: 6,146
Transactions rows: 13,305,915


In [6]:
output_paths = {
    "users": BRONZE_DIR / f"bronze_users_{snapshot_date}.csv",
    "cards": BRONZE_DIR / f"bronze_cards_{snapshot_date}.csv",
    "transactions": BRONZE_DIR / f"bronze_transactions_{snapshot_date}.csv",
    "mcc_codes": BRONZE_DIR / f"bronze_mcc_codes_{snapshot_date}.json",
    "fraud_labels": BRONZE_DIR / f"bronze_fraud_labels_{snapshot_date}.json",
}

users_df.to_csv(output_paths["users"], index=False)
cards_df.to_csv(output_paths["cards"], index=False)
transactions_df.to_csv(output_paths["transactions"], index=False)

with output_paths["mcc_codes"].open("w", encoding="utf-8") as file:
    json.dump(mcc_data, file, indent=2, ensure_ascii=False)

with output_paths["fraud_labels"].open("w", encoding="utf-8") as file:
    json.dump(fraud_labels_data, file, indent=2, ensure_ascii=False)

print("Bronze ingestion complete. Outputs:")
for name, path in output_paths.items():
    print(f" - {name:12} | {path}")

Bronze ingestion complete. Outputs:
 - users        | C:\Users\whale\BSc\DSPP\Data\bronze\bronze_users_20260825.csv
 - cards        | C:\Users\whale\BSc\DSPP\Data\bronze\bronze_cards_20260825.csv
 - transactions | C:\Users\whale\BSc\DSPP\Data\bronze\bronze_transactions_20260825.csv
 - mcc_codes    | C:\Users\whale\BSc\DSPP\Data\bronze\bronze_mcc_codes_20260825.json
 - fraud_labels | C:\Users\whale\BSc\DSPP\Data\bronze\bronze_fraud_labels_20260825.json
